In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from utils import *
from scipy.ndimage import gaussian_filter

In [2]:
dark_file = '/home/ulyanov/data/solo/phi/dark/solo_CAL1_phi-fdt-dark_20240205T033810_V202402220119C_0422051001.fits.gz'

with fits.open(dark_file) as hdul:
    dark = hdul[0].data

In [10]:
folder = '/home/ulyanov/data/solo/phi/linescans/'
files = sorted(glob.glob(folder + '*.fits.gz'))
files

['/home/ulyanov/data/solo/phi/linescans/solo_CAL1_phi-fdt-wavescan_20240309T100255_V202604251153C_0463090850.fits.gz',
 '/home/ulyanov/data/solo/phi/linescans/solo_CAL1_phi-fdt-wavescan_20240309T100709_V202604251153C_0463090851.fits.gz',
 '/home/ulyanov/data/solo/phi/linescans/solo_CAL1_phi-fdt-wavescan_20240309T101119_V202604251155C_0463090852.fits.gz',
 '/home/ulyanov/data/solo/phi/linescans/solo_CAL1_phi-fdt-wavescan_20240309T101529_V202604251155C_0463090853.fits.gz']

In [11]:
data_ = []

for file in files:
    with fits.open(file) as hdul:
        header = hdul[0].header
        data_.append(hdul[0].data[25:] - 0.4 * crop(dark, header))

data_ = np.array(data_).transpose((1,0,2,3))
data_ = demodulate(data_)

wv_ = read_wavelengths(header)[25:]

y0, x0 = header['PXBEG1'] - 1, header['PXBEG2'] - 1
nx, ny = data_.shape[-2:]


In [12]:
temp = (data_[:-9,2] - data_[9:,2]) / (data_[:-9,0] + data_[9:,0])
wv_ = wv_[:-9].copy()

In [13]:
folder = '/home/ulyanov/data/solo/phi/test/'
#folder = '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2026-03-10/'
files = sorted(glob.glob(folder + '*.fits.gz'))
files

['/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240101T040003_V202401090117C_0441010503.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240106T210007_V202401100517C_0441060508.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240107T000009_V202401110118C_0441070501.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240207T000009_V202402130123C_0442070501.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240207T060009_V202402130123C_0442070502.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240318T190009_V202405151841C_0443180504.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240328T060009_V202405152307C_0443281521.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240330T044009_V202405152319C_0443301531.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240330T233009_V202405152329C_0443300501.fits.gz',
 '/home/ulyanov/data/solo/ph

In [99]:
with fits.open(files[-7]) as hdul:
    header = hdul[0].header
    data = hdul[0].data - 0.4 * crop(dark, header)

nx, ny = data.shape[-2:]
y1, x1 = header['PXBEG1'] - 1, header['PXBEG2'] - 1

cpos = header['CONTPOS'] - 1
wv = read_wavelengths(header)
data = data.reshape(6,4,nx,ny)
data = demodulate(data)

print(cpos)

5


In [100]:
i = -1

plt.figure(figsize=(10,10))
plt.imshow(data[i,2], vmin=-50, vmax=50)
plt.tight_layout()

In [104]:
print(wv[i] - wv_[0], (wv[i] - wv_[0]) % 0.27)

q = interpolate(temp, wv_ - wv_[0], np.expand_dims([(wv[i] - wv_[0]) % 0.27], (1,2)))[0]
q = q[x1-x0:x1-x0+nx,y1-y0:y1-y0+ny]

plt.figure(figsize=(10,10))
#plt.imshow(q, vmin=-1e-3, vmax=1e-3)
plt.imshow(data[i,2] - q * data[-1,0], vmin=-50, vmax=50)
plt.tight_layout()

0.5169999999998254 0.24699999999982536


In [60]:
(wv[i] - wv_[0]) % 0.27

np.float64(0.026999999999534285)